# Day 9: 3D Descriptors + Chemprop Comparative Analysis

**Problem:** Day 8 GNN+LGBM got 0.5639 CV RAE but 0.68 blind test - same as baseline LGBM.

**Hypothesis:** 
- 2D features (Morgan FP + RDKit) miss 3D binding geometry
- Custom GNN trained from scratch on small data can't beat LGBM
- Pre-trained Chemprop (state-of-art MPNN) should help
- 3D conformer descriptors capture shape/volume for PXR's large binding pocket

**Strategy:**
1. Extract 3D conformer descriptors (volume, shape, surface area)
2. Train Chemprop (pre-trained message passing network)
3. Comparative analysis:
   - Baseline: LGBM with RDKit + Morgan FP
   - LGBM + 3D descriptors
   - Chemprop alone
   - Ensemble: LGBM + Chemprop + 3D features
4. Generate submission with best approach

**Target: 0.60-0.62 RAE**

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

# RDKit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Descriptors3D
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdMolDescriptors import CalcPBF
import useful_rdkit_utils as uru

# ML
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
tqdm.pandas()
sns.set_style("whitegrid")

print("✅ Imports complete")

## 1. Load Data

In [ ]:
train_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv")
test_df = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")

print(f"Train: {len(train_df)} molecules")
print(f"Test: {len(test_df)} molecules")
print(f"\nTarget distribution:")
print(train_df['pEC50'].describe())

## 2. Feature Generation Functions

In [ ]:
def get_3d_conformer_features(smiles, optimize=True, random_seed=42):
    """
    Generate 3D conformer and extract shape/volume descriptors.
    
    Key for PXR: Large binding pocket (~1150 Å³) means molecular volume,
    shape, and hydrophobic surface area are critical for binding.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(15)
    
    try:
        # Add hydrogens (needed for accurate 3D geometry)
        mol = Chem.AddHs(mol)
        
        # Generate 3D conformer
        result = AllChem.EmbedMolecule(mol, randomSeed=random_seed)
        if result != 0:  # Embedding failed
            return np.zeros(15)
        
        # Optimize geometry with MMFF force field
        if optimize:
            AllChem.MMFFOptimizeMolecule(mol)
        
        # Extract 3D descriptors
        features = [
            # Principal Moments of Inertia (PMI) - molecular shape
            Descriptors3D.PMI1(mol),  # Smallest PMI
            Descriptors3D.PMI2(mol),  # Middle PMI
            Descriptors3D.PMI3(mol),  # Largest PMI
            
            # Normalized PMI Ratios (shape descriptors)
            # NPR1 near 0.5 = rod, near 1.0 = disk
            # NPR2 near 0.5 = disk, near 1.0 = sphere
            Descriptors3D.NPR1(mol),
            Descriptors3D.NPR2(mol),
            
            # Shape measures
            Descriptors3D.RadiusOfGyration(mol),  # Size measure
            Descriptors3D.InertialShapeFactor(mol),  # Shape anisotropy
            Descriptors3D.Asphericity(mol),  # Deviation from sphere
            Descriptors3D.Eccentricity(mol),  # Elongation
            Descriptors3D.SpherocityIndex(mol),  # How sphere-like
            
            # Surface and volume
            AllChem.ComputeMolVolume(mol, confId=0),  # Molecular volume
            Descriptors3D.ASA(mol),  # Accessible surface area
            Descriptors3D.TPSA(mol),  # 3D topological polar surface area
            
            # Planarity
            CalcPBF(mol),  # Plane of Best Fit (planarity measure)
            
            # Complexity
            len([a for a in mol.GetAtoms()])  # Number of atoms (with H)
        ]
        
        # Remove hydrogens before returning
        mol = Chem.RemoveHs(mol)
        
        return np.array(features, dtype=float)
    
    except Exception as e:
        # If conformer generation fails, return zeros
        return np.zeros(15)

# Test function
test_smiles = train_df['SMILES'].iloc[0]
test_features = get_3d_conformer_features(test_smiles)
print(f"\n3D feature vector shape: {test_features.shape}")
print(f"Example features: {test_features[:5]}")
print("\n3D Feature names:")
feature_3d_names = [
    'PMI1', 'PMI2', 'PMI3', 'NPR1', 'NPR2',
    'RadiusOfGyration', 'InertialShapeFactor', 'Asphericity', 
    'Eccentricity', 'SpherocityIndex',
    'MolVolume', 'ASA', 'TPSA_3D', 'PlaneBestFit', 'NumAtoms_withH'
]
print(feature_3d_names)

In [ ]:
def get_scaffold(smiles):
    """Get Bemis-Murcko scaffold."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return "invalid"
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        return scaffold
    except:
        return "invalid"

def calculate_rae(y_true, y_pred):
    """Calculate Relative Absolute Error."""
    numerator = np.sum(np.abs(y_true - y_pred))
    denominator = np.sum(np.abs(y_true - np.mean(y_true)))
    return numerator / denominator if denominator != 0 else np.inf

## 3. Generate All Features

In [ ]:
print("Generating features for training set...\n")

# 1. RDKit descriptors (baseline)
print("1/4: RDKit descriptors...")
rdkit_desc = uru.RDKitDescriptors()
train_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(train_df.SMILES)]
test_df["rdkit_desc"] = [rdkit_desc.calc_smiles(x) for x in tqdm(test_df.SMILES)]

# 2. Morgan fingerprints (baseline)
print("\n2/4: Morgan fingerprints...")
def get_morgan_fp(smiles, radius=2, nbits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits))

train_df["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(train_df.SMILES)]
test_df["morgan_fp"] = [get_morgan_fp(x) for x in tqdm(test_df.SMILES)]

# 3. 3D conformer descriptors (NEW!)
print("\n3/4: 3D conformer descriptors (this takes ~5-10 min)...")
train_df["features_3d"] = [get_3d_conformer_features(x) for x in tqdm(train_df.SMILES)]
test_df["features_3d"] = [get_3d_conformer_features(x) for x in tqdm(test_df.SMILES)]

# 4. Scaffolds for grouped CV
print("\n4/4: Computing scaffolds...")
train_df["scaffold"] = [get_scaffold(s) for s in tqdm(train_df.SMILES)]
scaffold_to_group = {s: i for i, s in enumerate(train_df["scaffold"].unique())}
train_df["scaffold_group"] = train_df["scaffold"].map(scaffold_to_group)

print(f"\n✅ Feature generation complete!")
print(f"   Unique scaffolds: {len(scaffold_to_group)}")

## 4. Prepare Feature Matrices

In [ ]:
# Preprocess RDKit descriptors
rdkit_train = np.stack(train_df.rdkit_desc.values)
rdkit_test = np.stack(test_df.rdkit_desc.values)

imputer = SimpleImputer(strategy='median')
rdkit_train = imputer.fit_transform(rdkit_train)
rdkit_test = imputer.transform(rdkit_test)

# Normalize features
scaler_rdkit = StandardScaler()
scaler_morgan = StandardScaler()
scaler_3d = StandardScaler()

rdkit_train_norm = scaler_rdkit.fit_transform(rdkit_train)
rdkit_test_norm = scaler_rdkit.transform(rdkit_test)

morgan_train = np.stack(train_df.morgan_fp.values)
morgan_test = np.stack(test_df.morgan_fp.values)
morgan_train_norm = scaler_morgan.fit_transform(morgan_train)
morgan_test_norm = scaler_morgan.transform(morgan_test)

features_3d_train = np.stack(train_df.features_3d.values)
features_3d_test = np.stack(test_df.features_3d.values)
features_3d_train_norm = scaler_3d.fit_transform(features_3d_train)
features_3d_test_norm = scaler_3d.transform(features_3d_test)

# Create feature combinations
X_baseline = np.hstack([rdkit_train_norm, morgan_train_norm])  # Baseline: RDKit + Morgan
X_with_3d = np.hstack([rdkit_train_norm, morgan_train_norm, features_3d_train_norm])  # + 3D

X_test_baseline = np.hstack([rdkit_test_norm, morgan_test_norm])
X_test_with_3d = np.hstack([rdkit_test_norm, morgan_test_norm, features_3d_test_norm])

y_train = train_df["pEC50"].values
scaffold_groups = train_df["scaffold_group"].values

print(f"Feature matrix shapes:")
print(f"  Baseline (RDKit + Morgan): {X_baseline.shape}")
print(f"  With 3D descriptors: {X_with_3d.shape}")
print(f"  3D features added: {features_3d_train_norm.shape[1]} dims")

## 5. Comparative Analysis: LGBM with Different Features

In [ ]:
# Model configuration
lgbm_params = {
    'n_estimators': 3000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 7,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'verbose': -1
}

group_kfold = GroupKFold(n_splits=5)

# Test configurations
configs = [
    ('Baseline: RDKit + Morgan', X_baseline, X_test_baseline),
    ('With 3D: RDKit + Morgan + 3D', X_with_3d, X_test_with_3d),
]

results = []

print("="*80)
print("COMPARATIVE ANALYSIS: LGBM WITH DIFFERENT FEATURE SETS")
print("="*80)

for config_name, X_train_config, X_test_config in configs:
    print(f"\n{'='*80}")
    print(f"Configuration: {config_name}")
    print(f"Features: {X_train_config.shape[1]} dimensions")
    print(f"{'='*80}")
    
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_train_config, y_train, scaffold_groups), 1):
        X_tr, X_val = X_train_config[train_idx], X_train_config[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        model = LGBMRegressor(**lgbm_params, random_state=fold)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        
        rae = calculate_rae(y_val, y_pred)
        r2 = r2_score(y_val, y_pred)
        mae = mean_absolute_error(y_val, y_pred)
        
        fold_results.append({'rae': rae, 'r2': r2, 'mae': mae})
        print(f"  Fold {fold}: RAE={rae:.4f}, R²={r2:.4f}, MAE={mae:.4f}")
    
    avg_rae = np.mean([r['rae'] for r in fold_results])
    std_rae = np.std([r['rae'] for r in fold_results])
    avg_r2 = np.mean([r['r2'] for r in fold_results])
    avg_mae = np.mean([r['mae'] for r in fold_results])
    
    print(f"\n  → CV RAE: {avg_rae:.4f} ± {std_rae:.4f}")
    print(f"  → CV R²: {avg_r2:.4f}")
    print(f"  → CV MAE: {avg_mae:.4f}")
    
    results.append({
        'config': config_name,
        'n_features': X_train_config.shape[1],
        'cv_rae': avg_rae,
        'cv_rae_std': std_rae,
        'cv_r2': avg_r2,
        'cv_mae': avg_mae
    })

results_df = pd.DataFrame(results)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(results_df.to_string(index=False))
print("\n" + "="*80)

# Calculate improvement
baseline_rae = results_df.loc[results_df['config'].str.contains('Baseline'), 'cv_rae'].values[0]
with_3d_rae = results_df.loc[results_df['config'].str.contains('3D'), 'cv_rae'].values[0]
improvement = (baseline_rae - with_3d_rae) / baseline_rae * 100

print(f"\n3D Features Impact: {improvement:+.2f}% change in RAE")
if improvement > 0:
    print(f"✅ 3D features IMPROVE performance!")
else:
    print(f"❌ 3D features do NOT improve performance")

## 6. Install and Test Chemprop

Chemprop is state-of-the-art for molecular property prediction. It uses directed message passing neural networks (D-MPNN) pre-trained on millions of molecules.

In [ ]:
# Install chemprop (uncomment if needed)
# !pip install chemprop -q

# Check if chemprop is available
try:
    import chemprop
    print(f"✅ Chemprop version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True
except ImportError:
    print("❌ Chemprop not installed. Install with: pip install chemprop")
    print("   Skipping Chemprop analysis...")
    CHEMPROP_AVAILABLE = False

In [ ]:
if CHEMPROP_AVAILABLE:
    from chemprop import data, train, models
    import tempfile
    import os
    
    print("Testing Chemprop with 5-fold CV...\n")
    print("Note: Chemprop training is slower but often more accurate for small datasets")
    print("Expected time: ~10-15 min for 5 folds\n")
    
    chemprop_fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_baseline, y_train, scaffold_groups), 1):
        print(f"Fold {fold}/5...")
        
        # Prepare data for Chemprop (needs CSV format)
        train_fold_df = pd.DataFrame({
            'smiles': train_df['SMILES'].iloc[train_idx],
            'pEC50': y_train[train_idx]
        })
        val_fold_df = pd.DataFrame({
            'smiles': train_df['SMILES'].iloc[val_idx],
            'pEC50': y_train[val_idx]
        })
        
        # Save to temp files
        with tempfile.TemporaryDirectory() as tmpdir:
            train_path = os.path.join(tmpdir, 'train.csv')
            val_path = os.path.join(tmpdir, 'val.csv')
            model_path = os.path.join(tmpdir, 'model')
            
            train_fold_df.to_csv(train_path, index=False)
            val_fold_df.to_csv(val_path, index=False)
            
            # Train Chemprop
            args = [
                '--data_path', train_path,
                '--separate_val_path', val_path,
                '--dataset_type', 'regression',
                '--save_dir', model_path,
                '--num_folds', '1',
                '--epochs', '30',
                '--batch_size', '50',
                '--hidden_size', '300',
                '--depth', '3',
                '--dropout', '0.1',
                '--quiet'
            ]
            
            args = train.TrainArgs().parse_args(args)
            mean_score, std_score = train.cross_validate(args=args, train_func=train.run_training)
            
            # Make predictions on validation set
            pred_args = [
                '--test_path', val_path,
                '--checkpoint_dir', model_path,
                '--preds_path', os.path.join(tmpdir, 'preds.csv')
            ]
            pred_args = train.PredictArgs().parse_args(pred_args)
            preds = train.make_predictions(args=pred_args)
            
            # Calculate metrics
            y_val = val_fold_df['pEC50'].values
            y_pred = np.array(preds).flatten()
            
            rae = calculate_rae(y_val, y_pred)
            r2 = r2_score(y_val, y_pred)
            mae = mean_absolute_error(y_val, y_pred)
            
            chemprop_fold_results.append({'rae': rae, 'r2': r2, 'mae': mae})
            print(f"  RAE={rae:.4f}, R²={r2:.4f}, MAE={mae:.4f}\n")
    
    # Summary
    chemprop_avg_rae = np.mean([r['rae'] for r in chemprop_fold_results])
    chemprop_std_rae = np.std([r['rae'] for r in chemprop_fold_results])
    chemprop_avg_r2 = np.mean([r['r2'] for r in chemprop_fold_results])
    
    print("="*80)
    print("CHEMPROP RESULTS")
    print("="*80)
    print(f"CV RAE: {chemprop_avg_rae:.4f} ± {chemprop_std_rae:.4f}")
    print(f"CV R²: {chemprop_avg_r2:.4f}")
    print("="*80)
    
    # Add to results
    chemprop_result = {
        'config': 'Chemprop (D-MPNN)',
        'n_features': 'learned',
        'cv_rae': chemprop_avg_rae,
        'cv_rae_std': chemprop_std_rae,
        'cv_r2': chemprop_avg_r2,
        'cv_mae': np.mean([r['mae'] for r in chemprop_fold_results])
    }
    results.append(chemprop_result)
    results_df = pd.DataFrame(results)
else:
    print("Skipping Chemprop (not installed)")
    chemprop_avg_rae = None

## 7. Ensemble: LGBM + Chemprop with Stacked Generalization

In [ ]:
if CHEMPROP_AVAILABLE:
    print("Creating ensemble with stacked generalization...\n")
    
    # Get out-of-fold predictions from LGBM and Chemprop
    # This ensures no data leakage when training meta-model
    
    lgbm_oof = np.zeros(len(y_train))
    chemprop_oof = np.zeros(len(y_train))
    
    print("Generating out-of-fold predictions for stacking...")
    
    for fold, (train_idx, val_idx) in enumerate(group_kfold.split(X_with_3d, y_train, scaffold_groups), 1):
        print(f"  Fold {fold}/5...")
        
        # LGBM predictions
        X_tr, X_val = X_with_3d[train_idx], X_with_3d[val_idx]
        y_tr = y_train[train_idx]
        
        lgbm = LGBMRegressor(**lgbm_params, random_state=fold)
        lgbm.fit(X_tr, y_tr)
        lgbm_oof[val_idx] = lgbm.predict(X_val)
        
        # Chemprop predictions (reuse from earlier)
        # In practice, we'd retrain, but for speed we'll use a simpler approach
    
    # For now, just use simple weighted average
    # In production, you'd train Ridge on [lgbm_oof, chemprop_oof] -> y_train
    
    print("\nNote: Full stacking implementation would train meta-learner on OOF predictions")
    print("For this analysis, we'll compare individual models and decide on best approach")
else:
    print("Skipping ensemble (Chemprop not available)")

## 8. Final Comparison Visualization

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RAE comparison
ax1 = axes[0]
y_pos = np.arange(len(results_df))
ax1.barh(y_pos, results_df['cv_rae'], xerr=results_df['cv_rae_std'],
         color=['steelblue', 'green', 'coral'][:len(results_df)], alpha=0.7)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(results_df['config'])
ax1.set_xlabel('Cross-Validation RAE')
ax1.set_title('Model Comparison: RAE')
ax1.axvline(0.68, color='red', linestyle='--', label='Day 8 blind test (0.68)', linewidth=2)
ax1.axvline(0.60, color='green', linestyle='--', label='Target (0.60)', linewidth=2, alpha=0.5)
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# R² comparison
ax2 = axes[1]
ax2.barh(y_pos, results_df['cv_r2'],
         color=['steelblue', 'green', 'coral'][:len(results_df)], alpha=0.7)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(results_df['config'])
ax2.set_xlabel('Cross-Validation R²')
ax2.set_title('Model Comparison: R²')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)
print(results_df.sort_values('cv_rae').to_string(index=False))
print("="*80)

## 9. Decision: Choose Best Model and Generate Submission

In [ ]:
# Select best model based on CV RAE
best_model_row = results_df.loc[results_df['cv_rae'].idxmin()]
best_model_name = best_model_row['config']
best_cv_rae = best_model_row['cv_rae']

print("="*80)
print("BEST MODEL SELECTION")
print("="*80)
print(f"Best model: {best_model_name}")
print(f"CV RAE: {best_cv_rae:.4f}")
print("="*80)

# Train final model on full training data
print(f"\nTraining final model on full training set...")

if '3D' in best_model_name:
    X_train_final = X_with_3d
    X_test_final = X_test_with_3d
    use_3d = True
else:
    X_train_final = X_baseline
    X_test_final = X_test_baseline
    use_3d = False

# Train final LGBM model
final_model = LGBMRegressor(**lgbm_params, random_state=42)
final_model.fit(X_train_final, y_train)

# Generate predictions
test_preds = final_model.predict(X_test_final)

print(f"\nTest predictions:")
print(f"  Mean: {test_preds.mean():.3f}")
print(f"  Std: {test_preds.std():.3f}")
print(f"  Range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")
print(f"\nTrain mean: {y_train.mean():.3f}")

# Analyze 3D feature importance if used
if use_3d:
    print("\n" + "="*80)
    print("3D FEATURE IMPORTANCE ANALYSIS")
    print("="*80)
    
    # Get feature importances
    all_feature_names = (
        [f"rdkit_{i}" for i in range(217)] +
        [f"morgan_{i}" for i in range(2048)] +
        feature_3d_names
    )
    
    importances = pd.Series(final_model.feature_importances_, index=all_feature_names)
    
    # Top 3D features
    top_3d_features = importances[feature_3d_names].sort_values(ascending=False)
    print("\nTop 5 3D features:")
    print(top_3d_features.head())
    
    # Overall importance of 3D features
    total_3d_importance = importances[feature_3d_names].sum()
    total_importance = importances.sum()
    pct_3d = (total_3d_importance / total_importance) * 100
    
    print(f"\n3D features contribute: {pct_3d:.2f}% of total feature importance")
    print(f"3D features are {len(feature_3d_names)}/{len(all_feature_names)} = {len(feature_3d_names)/len(all_feature_names)*100:.1f}% of features")
    
    if pct_3d > 1.0:
        print("✅ 3D features are making meaningful contributions!")
    else:
        print("⚠️  3D features have low importance - may not be helping much")

## 10. Create Submission

In [ ]:
# Create submission
submission_df = pd.DataFrame({
    'SMILES': test_df['SMILES'],
    'Molecule Name': test_df['Molecule Name'],
    'pEC50': test_preds
})

# Determine output filename based on best model
if use_3d:
    output_path = '../outputs/day9_lgbm_with_3d_submission.csv'
else:
    output_path = '../outputs/day9_lgbm_baseline_submission.csv'

submission_df.to_csv(output_path, index=False)

print(f"✅ Submission saved to: {output_path}")
print(f"Shape: {submission_df.shape}")
print("\nFirst 10 predictions:")
print(submission_df.head(10))

# Visualize prediction distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(test_preds, bins=30, alpha=0.7, color='steelblue', edgecolor='black', label='Test predictions')
ax.hist(y_train, bins=30, alpha=0.5, color='coral', edgecolor='black', label='Train distribution')
ax.axvline(y_train.mean(), color='red', linestyle='--', linewidth=2, label='Train mean')
ax.set_xlabel('pEC50')
ax.set_ylabel('Count')
ax.set_title('Test Predictions vs Training Distribution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Validate Submission

In [ ]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from validation.activity_validation import validate_activity_submission

expected_activity_ids = set(test_df["Molecule Name"])
is_valid, validation_errors = validate_activity_submission(
    Path(output_path),
    expected_ids=expected_activity_ids,
)

if is_valid:
    print("✅ Submission file is valid!")
else:
    print("❌ Submission file is invalid:")
    for msg in validation_errors:
        print(f" - {msg}")

## Summary

**Tested approaches:**
1. Baseline: LGBM with RDKit + Morgan fingerprints
2. LGBM with 3D conformer descriptors added
3. Chemprop (state-of-art D-MPNN) [if installed]

**Key findings:**
- 3D descriptors capture molecular volume/shape relevant for PXR's large binding pocket
- Best model selected based on cross-validation RAE
- Check if 3D features made meaningful contribution via feature importance

**Next steps if RAE still > 0.60:**
- Ensemble best models with stacked generalization
- Add pharmacophore features (hydrophobic, aromatic counts)
- Augment with ChEMBL PXR data
- Try 3D-aware GNN (SchNet, DimeNet)
- Investigate CV-to-test performance gap